In [1]:
from pathlib import Path
import pandas as pd

notebook_dir = Path.cwd()
if notebook_dir.name != "notebooks":
    notebook_dir = notebook_dir / "notebooks"
dataset_path = notebook_dir.parent / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
# Keep the raw text so whitespace-only values can be checked before cleaning.
raw_df = pd.read_csv(dataset_path, keep_default_na=False)
df = raw_df.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(f"Dataset: {dataset_path}")
df.head()

Dataset: D:\customer-churn-prediction\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Kiểm tra missing values trong `TotalCharges`

Các giá trị chỉ gồm whitespace trong dữ liệu raw được xem là missing trước khi chuyển `TotalCharges` sang kiểu số.

In [2]:
# 1. Lọc và hiển thị các dòng TotalCharges missing.
missing_totalcharges = df[df["TotalCharges"].isna()]
missing_columns = ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]
print(f"Số dòng TotalCharges missing: {len(missing_totalcharges)}")
display(missing_totalcharges[missing_columns])

# 2. Kiểm tra dtype của TotalCharges sau khi chuyển đổi.
print(f"dtype của TotalCharges: {df['TotalCharges'].dtype}")

# 3. Kiểm tra whitespace trong dữ liệu raw trước khi chuyển đổi.
whitespace_mask = raw_df["TotalCharges"].astype(str).str.strip().eq("")
print(f"Số giá trị TotalCharges chỉ gồm whitespace: {whitespace_mask.sum()}")
print(f"Có whitespace-only values không: {whitespace_mask.any()}")

Số dòng TotalCharges missing: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


dtype của TotalCharges: float64
Số giá trị TotalCharges chỉ gồm whitespace: 11
Có whitespace-only values không: True


## 1. Rows và columns

Dataset có **7,043 rows** và **21 columns**.

In [3]:
rows, columns = df.shape
print(f"Rows: {rows:,}")
print(f"Columns: {columns}")

Rows: 7,043
Columns: 21


## 2. Features

Có 20 features; `Churn` là target variable.

In [4]:
target = "Churn"
features = [column for column in df.columns if column != target]
print(f"Features ({len(features)}):")
print(", ".join(features))

Features (20):
customerID, gender, SeniorCitizen, Partner, Dependents, tenure, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges


## 3. Numerical và categorical features

In [5]:
numerical_features = df[features].select_dtypes(include="number").columns.tolist()
categorical_features = df[features].select_dtypes(exclude="number").columns.tolist()
print(f"Numerical ({len(numerical_features)}): {', '.join(numerical_features)}")
print(f"Categorical ({len(categorical_features)}): {', '.join(categorical_features)}")

Numerical (4): SeniorCitizen, tenure, MonthlyCharges, TotalCharges
Categorical (16): customerID, gender, Partner, Dependents, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, PaperlessBilling, PaymentMethod


In [ ]:
categorical_columns = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "Churn",
]

for column in categorical_columns:
    counts = df[column].value_counts(dropna=False)
    summary = pd.DataFrame(
        {
            "count": counts,
            "percentage": (counts / len(df) * 100).round(2),
        }
    )
    summary.index.name = column
    display(summary)


In [ ]:
distribution_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen",
]

numerical_summary = df[distribution_columns].describe()
display(numerical_summary)

,tenure,MonthlyCharges,TotalCharges,SeniorCitizen
count,7043.000000,7043.000000,7032.000000,7043.000000
mean,32.371149,64.761692,2283.300441,0.162147
std,24.559481,30.090047,2266.771362,0.368612
min,0.000000,18.250000,18.800000,0.000000
25%,9.000000,35.500000,401.450000,0.000000
50%,29.000000,70.350000,1397.475000,0.000000
75%,55.000000,89.850000,3794.737500,0.000000
max,72.000000,118.750000,8684.800000,1.000000


## 4. Target variable

Target variable là **`Churn`**, với hai giá trị `Yes` và `No`.

In [7]:
print(f"Target: {target}")
print(f"Values: {df[target].dropna().unique().tolist()}")

Target: Churn
Values: ['No', 'Yes']


## 5. Missing values

In [8]:
missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
if missing_values.empty:
    print("Không có missing values.")
else:
    print(missing_values.to_string())

TotalCharges    11


## 6. Duplicate records

In [9]:
unique_customer_ids = df["customerID"].nunique()
total_rows = len(df)
duplicate_customer_ids = df["customerID"].duplicated().sum()

print(f"Unique customerID: {unique_customer_ids:,}")
print(f"Total rows: {total_rows:,}")
print(f"Duplicate customerID: {duplicate_customer_ids:,}")

Unique customerID: 7,043
Total rows: 7,043
Duplicate customerID: 0


## 7. Churn và non-churn rate

In [10]:
churn_summary = df[target].value_counts().rename_axis(target).reset_index(name="count")
churn_summary["percentage"] = churn_summary["count"] / len(df) * 100
print(churn_summary.to_string(index=False))
for _, row in churn_summary.iterrows():
    label = "Churn" if row[target] == "Yes" else "Non-churn"
    print(f"{label}: {row['count']:,} ({row['percentage']:.2f}%)")

Churn  count  percentage
   No   5174   73.463013
  Yes   1869   26.536987
Non-churn: 5,174 (73.46%)
Churn: 1,869 (26.54%)


## Summary

- 7,043 rows, 21 columns, 20 features.
- Numerical: `SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`.
- Categorical: các features còn lại; `customerID` là identifier.
- Target: `Churn`.
- Có 11 missing values trong `TotalCharges`.
- Có 0 duplicate records.
- Non-churn: 5,174 (73.46%); churn: 1,869 (26.54%).